In [ ]:
#%pip install --upgrade pip
#%pip install xgboost catboost lightgbm 

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----------- ---------------------------- 0.5/1.8 MB 5.7 MB/s eta 0:00:01
   ---------------------------------- ----- 1.6/1.8 MB 6.7 MB/s eta 0:00:01
   ---------------------------------- ----- 1.6/1.8 MB 6.7 MB/s eta 0:00:01
   ---------------------------------- ----- 1.6/1.8 MB 6.7 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 1.9 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from preprocessing import train_clean, test_clean
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier 
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB 
from sklearn.metrics import accuracy_score 

In [9]:
X = train_clean.drop(columns = ['Survived'])
y = train_clean['Survived']

n_spl = 5
skf = StratifiedKFold(n_splits = n_spl, shuffle = True, random_state = 42)

In [ ]:
#Linear regression
fold_accuracy = {
    'Linear Regression': [],
    'Ridge (L2)': [],
    'Lasso (L1)': [],
    'ElasticNet (L1 + L2)': []
}

for fold, (train_idx, val_idx) in enumerate (skf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    
    models = {
        'Linear Regression': LinearRegression(),
        'Ridge (L2)': Ridge(alpha=0.01),
        'Lasso (L1)': Lasso(alpha=0.01),
        'ElasticNet (L1 + L2)': ElasticNet(alpha=0.01, l1_ratio = 0.5)
    }
    
    for name, model in models.items():
        model.fit(X_train_scaled, y_train)
        y_pr = model.predict(X_val_scaled)
        y_pred = (y_pr >= 0.5).astype(int)
        acc = accuracy_score(y_val, y_pred)
        fold_accuracy[name].append(acc)
        print(f'{name:<20} Accuracy: {acc:.4f}')

for name, accuracies in fold_accuracy.items():   
    print(f'{name:<20} Mean Accuracy: {np.mean(accuracies):.4f}')

Linear Regression    Accuracy: 0.8324
Ridge (L2)           Accuracy: 0.8324
Lasso (L1)           Accuracy: 0.8101
ElasticNet (L1 + L2) Accuracy: 0.8212
Linear Regression    Accuracy: 0.8146
Ridge (L2)           Accuracy: 0.8146
Lasso (L1)           Accuracy: 0.8146
ElasticNet (L1 + L2) Accuracy: 0.8090
Linear Regression    Accuracy: 0.8315
Ridge (L2)           Accuracy: 0.8315
Lasso (L1)           Accuracy: 0.8258
ElasticNet (L1 + L2) Accuracy: 0.8258
Linear Regression    Accuracy: 0.8258
Ridge (L2)           Accuracy: 0.8258
Lasso (L1)           Accuracy: 0.8146
ElasticNet (L1 + L2) Accuracy: 0.8146
Linear Regression    Accuracy: 0.8427
Ridge (L2)           Accuracy: 0.8427
Lasso (L1)           Accuracy: 0.8258
ElasticNet (L1 + L2) Accuracy: 0.8371
Linear Regression    Mean Accuracy: 0.8294
Ridge (L2)           Mean Accuracy: 0.8294
Lasso (L1)           Mean Accuracy: 0.8182
ElasticNet (L1 + L2) Mean Accuracy: 0.8215


In [ ]:
#Kneighbors
n_neighbors_list = [3, 5, 7, 9, 11, 13, 15]
weights_list = ['uniform', 'distance']
metric_lists = ['euclidean', 'manhattan']
best_accuracy = 0.0
best_params = {}
for n in n_neighbors_list:
    for w in weights_list:
        for m in metric_lists:
            fold_accuracies = []
            for train_idx, val_idx in skf.split(X, y):
                X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
                X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
                scaler = StandardScaler()
                X_train_scaled = scaler.fit_transform(X_train)
                X_val_scaled = scaler.transform(X_val)
    
                model = KNeighborsClassifier(
                    n_neighbors = n,
                    weights = w,
                    metric = m
                )
                model.fit(X_train_scaled, y_train)
                y_pred = model.predict(X_val_scaled)
                acc = accuracy_score(y_val, y_pred)
                fold_accuracies.append(acc)
            mean_acc = np.mean(fold_accuracies)
            print(f'k = {n:<2} | weights = "{w:<8}" | metric = "{m:<9}" -> Mean Accuracy: {mean_acc:.4f}') 
            if mean_acc > best_accuracy:
                best_accuracy = mean_acc
                best_params = {
                    'n_neighbors': n,
                    'weights': w,
                    'metric': m
                }
print(f'Лучший результат: {best_accuracy}')
print(f'Параметры: {best_params}')


k = 3  | weights = "uniform " | metric = "euclidean" -> Mean Accuracy: 0.8103
k = 3  | weights = "uniform " | metric = "manhattan" -> Mean Accuracy: 0.8092
k = 3  | weights = "distance" | metric = "euclidean" -> Mean Accuracy: 0.7901
k = 3  | weights = "distance" | metric = "manhattan" -> Mean Accuracy: 0.7923
k = 5  | weights = "uniform " | metric = "euclidean" -> Mean Accuracy: 0.8373
k = 5  | weights = "uniform " | metric = "manhattan" -> Mean Accuracy: 0.8339
k = 5  | weights = "distance" | metric = "euclidean" -> Mean Accuracy: 0.8125
k = 5  | weights = "distance" | metric = "manhattan" -> Mean Accuracy: 0.8126
k = 7  | weights = "uniform " | metric = "euclidean" -> Mean Accuracy: 0.8283
k = 7  | weights = "uniform " | metric = "manhattan" -> Mean Accuracy: 0.8361
k = 7  | weights = "distance" | metric = "euclidean" -> Mean Accuracy: 0.8204
k = 7  | weights = "distance" | metric = "manhattan" -> Mean Accuracy: 0.8193
k = 9  | weights = "uniform " | metric = "euclidean" -> Mean Acc

In [ ]:
#Decision Tree
for fold, (train_idx, val_idx) in enumerate (skf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    
    model = DecisionTreeClassifier()
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_val_scaled)
    acc = accuracy_score(y_val, y_pred)
    fold_accuracy.append(acc)
    print(f'Accuracy: {acc:.4f}')
 
print(f'Mean Accuracy: {np.mean(fold_accuracy):.4f}')

Accuracy: 0.8268
Accuracy: 0.8034
Accuracy: 0.7584
Accuracy: 0.7640
Accuracy: 0.8090
Mean Accuracy: 0.7935


In [ ]:
#Random forest
fold_accuracy = []
best_accuracy = 0.0
best_param = 0
for n_est in range(100, 500, 10): 
    fold_accuracy = []
    for fold, (train_idx, val_idx) in enumerate (skf.split(X, y)):
        X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
        
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)
        
        model = RandomForestClassifier(n_estimators = n_est)
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_val_scaled)
        acc = accuracy_score(y_val, y_pred)
        fold_accuracy.append(acc)
    mean_acc = np.mean(fold_accuracy)
    print(f'"{n_est:<2}" -> Mean accuracy = {mean_acc:.4f}')
    
    if mean_acc > best_accuracy:
        best_accuracy = mean_acc
        best_param = n_est

print(f'Лучший результат: {best_accuracy}')
print(f'Параметр: {best_param}')

"100" -> Mean accuracy = 0.8283
"110" -> Mean accuracy = 0.8238
"120" -> Mean accuracy = 0.8249
"130" -> Mean accuracy = 0.8249
"140" -> Mean accuracy = 0.8283
"150" -> Mean accuracy = 0.8226
"160" -> Mean accuracy = 0.8283
"170" -> Mean accuracy = 0.8294
"180" -> Mean accuracy = 0.8260
"190" -> Mean accuracy = 0.8283
"200" -> Mean accuracy = 0.8271
"210" -> Mean accuracy = 0.8305
"220" -> Mean accuracy = 0.8249
"230" -> Mean accuracy = 0.8327
"240" -> Mean accuracy = 0.8215
"250" -> Mean accuracy = 0.8271
"260" -> Mean accuracy = 0.8328
"270" -> Mean accuracy = 0.8294
"280" -> Mean accuracy = 0.8249
"290" -> Mean accuracy = 0.8271
"300" -> Mean accuracy = 0.8294
"310" -> Mean accuracy = 0.8260
"320" -> Mean accuracy = 0.8260
"330" -> Mean accuracy = 0.8339
"340" -> Mean accuracy = 0.8271
"350" -> Mean accuracy = 0.8260
"360" -> Mean accuracy = 0.8316
"370" -> Mean accuracy = 0.8294
"380" -> Mean accuracy = 0.8249
"390" -> Mean accuracy = 0.8294
"400" -> Mean accuracy = 0.8294
"410" ->

In [ ]:
#XGBoost
fold_accuracy = []

for fold, (train_idx, val_idx) in enumerate (skf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    
    model = XGBClassifier(
        n_estimators = 200,
        learning_rate = 0.05,
        max_depth = 4,
        random_state = 42
    )
    
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_val_scaled)
    acc = accuracy_score(y_val, y_pred)
    fold_accuracy.append(acc)
    print(f'Accuracy: {acc:.4f}')
 
print(f'Mean Accuracy: {np.mean(fold_accuracy):.4f}')

Accuracy: 0.8547
Accuracy: 0.8427
Accuracy: 0.8090
Accuracy: 0.8258
Accuracy: 0.8315
Mean Accuracy: 0.8327


In [ ]:
#CatBoost
fold_accuracy = []

for fold, (train_idx, val_idx) in enumerate (skf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    
    model = CatBoostClassifier(
        iterations = 200,
        learning_rate = 0.05,
        depth = 4,
        verbose = 0,
        random_state = 42
    )
    
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_val_scaled)
    acc = accuracy_score(y_val, y_pred)
    fold_accuracy.append(acc)
    print(f'Accuracy: {acc:.4f}')
 
print(f'Mean Accuracy: {np.mean(fold_accuracy):.4f}')

Accuracy: 0.8547
Accuracy: 0.8371
Accuracy: 0.8258
Accuracy: 0.8202
Accuracy: 0.8539
Mean Accuracy: 0.8384


In [ ]:
#LGBMBoost
fold_accuracy = []

for fold, (train_idx, val_idx) in enumerate (skf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    
    model = LGBMClassifier(
        n_estimators = 200,
        learning_rate = 0.05,
        num_leaves = 15,
        max_depth = 4,
        random_state = 42,
        verbose = -1
    )
    
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_val_scaled)
    acc = accuracy_score(y_val, y_pred)
    fold_accuracy.append(acc)
    print(f'Accuracy: {acc:.4f}')
 
print(f'Mean Accuracy: {np.mean(fold_accuracy):.4f}')

Accuracy: 0.8603
Accuracy: 0.8427
Accuracy: 0.7921
Accuracy: 0.8427
Accuracy: 0.8258
Mean Accuracy: 0.8327


In [ ]:
#MLP
fold_accuracy = []

for fold, (train_idx, val_idx) in enumerate (skf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    
    model = MLPClassifier(
        hidden_layer_sizes = (32, 16),
        activation = 'relu',
        max_iter = 1000,
        random_state = 42
    )
    
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_val_scaled)
    acc = accuracy_score(y_val, y_pred)
    fold_accuracy.append(acc)
    print(f'Accuracy: {acc:.4f}')
 
print(f'Mean Accuracy: {np.mean(fold_accuracy):.4f}')

Accuracy: 0.8492
Accuracy: 0.8034
Accuracy: 0.7978
Accuracy: 0.8202
Accuracy: 0.8315
Mean Accuracy: 0.8204


C:\Users\User\AppData\Roaming\Python\Python313\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


In [ ]:
#Logistic regression
fold_accuracy = []

for fold, (train_idx, val_idx) in enumerate (skf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    
    model = LogisticRegression(
        max_iter = 1000,
        random_state = 42
    )
    
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_val_scaled)
    acc = accuracy_score(y_val, y_pred)
    fold_accuracy.append(acc)
    print(f'Accuracy: {acc:.4f}')
 
print(f'Mean Accuracy: {np.mean(fold_accuracy):.4f}')

Accuracy: 0.8268
Accuracy: 0.8146
Accuracy: 0.8315
Accuracy: 0.8315
Accuracy: 0.8315
Mean Accuracy: 0.8272


In [ ]:
#SVC
fold_accuracy = []

for fold, (train_idx, val_idx) in enumerate (skf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    
    model = SVC(
        kernel = 'rbf',
        C = 1.0,
        random_state = 42
    )
    
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_val_scaled)
    acc = accuracy_score(y_val, y_pred)
    fold_accuracy.append(acc)
    print(f'Accuracy: {acc:.4f}')
 
print(f'Mean Accuracy: {np.mean(fold_accuracy):.4f}')

Accuracy: 0.8380
Accuracy: 0.8146
Accuracy: 0.8258
Accuracy: 0.8427
Accuracy: 0.8539
Mean Accuracy: 0.8350


In [53]:
#GaussianNB
fold_accuracy = []

for fold, (train_idx, val_idx) in enumerate (skf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    
    model = GaussianNB()
    
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_val_scaled)
    acc = accuracy_score(y_val, y_pred)
    fold_accuracy.append(acc)
    print(f'Accuracy: {acc:.4f}')
 
print(f'Mean Accuracy: {np.mean(fold_accuracy):.4f}')

Accuracy: 0.7877
Accuracy: 0.7697
Accuracy: 0.7921
Accuracy: 0.7978
Accuracy: 0.8258
Mean Accuracy: 0.7946
